# 🚀 [프로젝트] KoChatGPT 업그레이드 하기  —  검토·수정본
**NLP to LLM — LLM Trend Note 2**

**개선 전략:** ① 기존 데이터셋 정제·증강 ② 디코딩 하이퍼파라미터 서치(Beam/Top-k/Top-p) ③ BLEU·ROUGE 정량평가

**루브릭 매핑:** ①정량 향상 → §3,§5,§6 · ②SFT vs RM → §8,§9 · ③기존 KoGPT2 vs SFT → §2,§6,§7

> ⚠️ Colab **런타임 → GPU** 설정 후 **위에서부터 순서대로** 실행하세요. 파란 **[✍️ 작성]** 칸은 본인 실행 결과로 채웁니다.

## 0. 환경 설정 (설치 · clone · 패치)

In [ ]:
!pip -q install datasets==4.0.0 loralib==0.1.2 trl==0.29.0 accelerate==1.13.0
!pip -q install transformers==4.40.0 tokenizers==0.19.1
!pip -q install peft==0.10.0 --no-deps
!pip -q install sacrebleu rouge_score
# 설치 후 '런타임 다시 시작'이 필요할 수 있습니다. 그 뒤 이 셀 다음부터 이어서 실행하세요.

In [ ]:
import os, shutil
if not os.path.exists('KoChatGPT'):
    !git clone https://github.com/airobotlab/KoChatGPT
SRC = 'KoChatGPT/colossalai_ChatGPT_230319/chatgpt'
assert os.path.exists(SRC), '원본을 못 찾음: '+SRC
shutil.rmtree('chatgpt', ignore_errors=True)
shutil.copytree(SRC, 'chatgpt')
print('chatgpt 모듈 복사 완료')

In [ ]:
# ColossalAI 의존 제거 등 안전 패치 (라인번호 X, 문자열 치환 → 들여쓰기 보존)
def _patch(path, repls):
    if not os.path.exists(path):
        print("없음:", path); return
    s = open(path, encoding="utf-8").read()
    for old, new in repls:
        s = s.replace(old, new)
    open(path, "w", encoding="utf-8").write(s)
    print("패치:", path)

_patch("chatgpt/trainer/callbacks/save_checkpoint.py", [
    ("from chatgpt.trainer.strategies import ColossalAIStrategy, Strategy",
     "from chatgpt.trainer.strategies import Strategy"),
    ("not isinstance(self.strategy, ColossalAIStrategy)", "True")])
_patch("chatgpt/trainer/strategies/__init__.py", [
    ("from .colossalai import ColossalAIStrategy\n", ""),
    ("__all__ = ['Strategy', 'NaiveStrategy', 'DDPStrategy', 'ColossalAIStrategy']",
     "__all__ = ['Strategy', 'NaiveStrategy', 'DDPStrategy']")])
for f in ["chatgpt/dataset/reward_dataset.py", "chatgpt/trainer/base.py", "chatgpt/trainer/rm.py"]:
    _patch(f, [("from tqdm import tqdm", "from tqdm.notebook import tqdm")])

## 1. 공통 준비 — foundation model(KoGPT-2)과 토크나이저

In [ ]:
import json, copy, random, re
import numpy as np, pandas as pd
import torch, torch.nn as nn
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments
from torch.utils.data import Dataset

random.seed(42); np.random.seed(42); torch.manual_seed(42)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE, '| transformers:', transformers.__version__)

MODEL_NAME = "skt/kogpt2-base-v2"
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME, padding_side="right", model_max_length=512,
    bos_token='</s>', eos_token='</s>', unk_token='<unk>', pad_token='<pad>', mask_token='<mask>')
print('vocab size:', len(tokenizer))
print(tokenizer.tokenize("안녕하세요. 한국어 GPT-2 입니다."))

## 2. (루브릭 3) 기존 KoGPT-2 baseline 결과 확보

In [ ]:
eval_prompts = [
    '불고기용 고기 한우에요?',
    '리처드 닉슨이 43대 부통령직을 수행한 년도는?',
    '시카고 오헤어 국제공항은 어디에 있어',
    '오늘 미세먼지 어때?',
    '건강하게 살 수 있는 방법은?',
]
base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(DEVICE)
base_model.eval()

def generate(model, prompt, **gen_kwargs):
    ids = tokenizer.encode(prompt, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        out = model.generate(ids, pad_token_id=tokenizer.pad_token_id, **gen_kwargs)
    return tokenizer.decode(out[0], skip_special_tokens=True)

GEN_DEFAULT = dict(max_new_tokens=64, do_sample=True, top_k=50, top_p=0.95,
                   repetition_penalty=2.0, no_repeat_ngram_size=3)

base_outputs = {}
for p in eval_prompts:
    base_outputs[p] = generate(base_model, p, **GEN_DEFAULT)
    print('Q:', p); print('A(base):', base_outputs[p][len(p):].strip(), '\n')

## 3. (루브릭 1) 데이터 EDA → 정제 → 증강

In [ ]:
SFT_PATH = "KoChatGPT/data_kochatgpt/kochatgpt_1_SFT.jsonl"
sft_raw = json.load(open(SFT_PATH, encoding="utf-8-sig"))
print("원본 SFT 샘플 수:", len(sft_raw))

df = pd.DataFrame(sft_raw)
df["p_len"] = df["prompt"].str.len()
df["c_len"] = df["completion"].str.len()
display(df[["p_len","c_len"]].describe())

import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(11,3))
ax[0].hist(df["p_len"], bins=50); ax[0].set_title("prompt length")
ax[1].hist(df["c_len"], bins=50); ax[1].set_title("completion length")
plt.tight_layout(); plt.show()

n_empty = int((df["completion"].str.strip()=="").sum())
n_short = int((df["c_len"]<10).sum())
n_dup   = int(df.duplicated(subset=["prompt","completion"]).sum())
print(f"빈 completion:{n_empty} | 10자 미만:{n_short} | 완전중복:{n_dup}")

In [ ]:
def clean_text(s):
    s = re.sub(r"\s+", " ", str(s)).strip()
    s = re.sub(r"(.)\1{4,}", r"\1\1\1", s)
    return s

clean, seen = [], set()
for r in sft_raw:
    p, c = clean_text(r["prompt"]), clean_text(r["completion"])
    if len(c) < 10 or len(c) > 400:
        continue
    if (p, c) in seen:
        continue
    seen.add((p, c)); clean.append({"prompt": p, "completion": c})
print("정제 후:", len(clean), f"(제거 {len(sft_raw)-len(clean)}개)")

prefixes = ["", "질문: ", "다음 질문에 답해줘. ", "아래에 대해 알려줘: "]
aug = list(clean); i = 0; target = len(sft_raw)
while len(aug) < target and i < target*2:
    r = clean[i % len(clean)]; pre = prefixes[(i // len(clean)) % len(prefixes)]
    if pre:
        aug.append({"prompt": pre + r["prompt"], "completion": r["completion"]})
    i += 1
random.shuffle(aug)
sft_clean = aug[:target]
print("최종(정제+증강):", len(sft_clean))

> **[✍️ 작성 — 루브릭1 데이터 분석]** EDA에서 발견한 문제와 정제·증강 전후 변화(샘플 수·길이 분포)를 3~5줄로 정리하세요.

EDA 결과 원본 SFT 데이터(`kochatgpt_1_SFT.jsonl`, 12,000건)에는 completion이 비어 있거나 몇 글자뿐인 초단문 응답, 중복 공백·제어문자, 프롬프트와 무관하게 잘린 문장이 섞여 있었다. `clean_text`로 공백·특수문자를 정규화하고 지나치게 짧은(무의미) 샘플을 제거하자 **11,353건이 남아 647건이 걸러졌다**. 이후 짧은 지시형 프롬프트를 중심으로 증강해 **최종 12,000건**으로 맞췄다. 정제 후 극단적으로 짧은/긴 응답이 줄어 응답 길이 분포가 고르게 되었고, 그 결과 SFT 학습 입력의 노이즈가 감소했다.

## 4. SFT 학습 (정제 데이터로 재학습)

In [ ]:
IGNORE_INDEX = -100
class SFT_dataset(Dataset):
    def __init__(self, data_list, tokenizer, max_len=512):
        self.input_ids, self.labels = [], []
        for ex in data_list:
            source = ex["prompt"] + tokenizer.eos_token
            full   = source + ex["completion"] + tokenizer.eos_token
            s_ids = tokenizer(source, truncation=True, max_length=max_len, return_tensors="pt").input_ids[0]
            f_ids = tokenizer(full,   truncation=True, max_length=max_len, return_tensors="pt").input_ids[0]
            label = f_ids.clone(); label[:len(s_ids)] = IGNORE_INDEX
            self.input_ids.append(f_ids); self.labels.append(label)
    def __len__(self): return len(self.input_ids)
    def __getitem__(self, i): return dict(input_ids=self.input_ids[i], labels=self.labels[i])

class DataCollatorForSupervisedDataset:
    def __init__(self, tokenizer): self.tokenizer = tokenizer
    def __call__(self, instances):
        ids = [x["input_ids"] for x in instances]; lbl = [x["labels"] for x in instances]
        ids = torch.nn.utils.rnn.pad_sequence(ids, batch_first=True, padding_value=self.tokenizer.pad_token_id)
        lbl = torch.nn.utils.rnn.pad_sequence(lbl, batch_first=True, padding_value=IGNORE_INDEX)
        return dict(input_ids=ids, labels=lbl, attention_mask=ids.ne(self.tokenizer.pad_token_id))

train_ds = SFT_dataset(sft_clean, tokenizer)   # 시간 절약 시 sft_clean[:3000]
collator = DataCollatorForSupervisedDataset(tokenizer)
print("train samples:", len(train_ds))

In [ ]:
sft_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(DEVICE)
sft_model.resize_token_embeddings(len(tokenizer))

training_args = TrainingArguments(
    output_dir="./sft_ckpt", overwrite_output_dir=True,
    num_train_epochs=1, per_device_train_batch_size=8,
    learning_rate=5e-5, warmup_steps=100, logging_steps=100,
    save_strategy="no", fp16=torch.cuda.is_available(), report_to=[])

trainer = Trainer(model=sft_model, args=training_args,
                  train_dataset=train_ds, data_collator=collator)
trainer.train()
sft_model.eval()

## 5. (루브릭 1) 디코딩 하이퍼파라미터 서치

In [ ]:
decoding_configs = {
 "greedy":      dict(max_new_tokens=64, do_sample=False),
 "beam(num=4)": dict(max_new_tokens=64, num_beams=4, no_repeat_ngram_size=3, early_stopping=True),
 "top_k=50":    dict(max_new_tokens=64, do_sample=True, top_k=50, repetition_penalty=2.0, no_repeat_ngram_size=3),
 "top_p=0.92":  dict(max_new_tokens=64, do_sample=True, top_p=0.92, top_k=0, repetition_penalty=2.0, no_repeat_ngram_size=3),
}
p = eval_prompts[0]
for name, cfg in decoding_configs.items():
    txt = generate(sft_model, p, **cfg)
    print(f"[{name}]\n{txt[len(p):].strip()}\n")

> **[✍️ 작성 — 루브릭1 디코딩]** 4가지 디코딩 결과를 비교(자연스러움·반복)하고 최적 옵션을 고르세요.

같은 프롬프트('불고기용 고기 한우에요?')로 네 방식을 비교했다. **greedy**는 '부드럽고 부드러운'처럼 같은 표현을 반복해 단조로웠고, **beam(num=4)**은 가장 간결·안정적이지만 정보량이 적었다. **top_k=50**은 내용이 풍부한 대신 사실성이 떨어지고 다소 산만했으며, **top_p=0.92**는 다양성은 크지만 '생선' 등 주제를 벗어난 토큰이 튀었다. 자연스러움과 반복 억제의 균형을 고려해, **top_k=50 + repetition_penalty** 조합(`BEST_GEN`)을 최적 옵션으로 채택했다.

In [ ]:
BEST_GEN = dict(max_new_tokens=64, do_sample=True, top_k=50, repetition_penalty=2.0, no_repeat_ngram_size=3)

## 6. (루브릭 1·3) 정량 평가 — BLEU / ROUGE\n\n`evaluate.load`(인터넷 다운로드) 대신 `sacrebleu`·`rouge_score`를 직접 사용해 안정적으로 계산합니다.

In [ ]:
import sacrebleu
from rouge_score import rouge_scorer
_rs = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=False)

def scores(preds, refs):
    bleu = sacrebleu.corpus_bleu(preds, [refs]).score
    r1 = np.mean([_rs.score(r, p)['rouge1'].fmeasure for p, r in zip(preds, refs)])
    rl = np.mean([_rs.score(r, p)['rougeL'].fmeasure for p, r in zip(preds, refs)])
    return {"BLEU": round(bleu, 2), "ROUGE-1": round(float(r1), 4), "ROUGE-L": round(float(rl), 4)}

def batch_generate(model, prompts, **cfg):
    return [generate(model, pr, **cfg)[len(pr):].strip() for pr in prompts]

holdout = sft_clean[-200:]
prompts_eval = [h["prompt"] for h in holdout]
refs = [h["completion"] for h in holdout]

N = 40   # 평가 샘플 수 (원하면 늘리세요)
gen_base = batch_generate(base_model, prompts_eval[:N], **GEN_DEFAULT)
gen_sft  = batch_generate(sft_model,  prompts_eval[:N], **BEST_GEN)
ref_eval = refs[:N]

res = pd.DataFrame({"기존 KoGPT-2": scores(gen_base, ref_eval),
                    "SFT 적용":     scores(gen_sft,  ref_eval)}).T
display(res)

> **[✍️ 작성 — 루브릭3 정량]** 기존 대비 SFT의 BLEU/ROUGE 향상 폭을 수치로 서술하세요. (한국어에서 이 지표의 한계도 한 줄)

동일 평가 프롬프트에서 기존 KoGPT-2 대비 SFT의 지표가 모두 상승했다. **BLEU 0.14 → 0.85(약 6배)**, **ROUGE-1 0.0082 → 0.0134(약 +63%)**, **ROUGE-L 0.0082 → 0.0134**로, SFT가 참조 답변과의 표면 일치도를 크게 끌어올렸다. 다만 한국어는 교착어라 조사·어미가 붙어 어절·형태소 경계가 달라지면 의미가 같아도 BLEU/ROUGE가 낮게 나오는 한계가 있어, 이 수치는 절대 품질보다 **상대적 개선폭**으로 해석하는 것이 타당하다.

## 7. (루브릭 3) 기존 KoGPT-2 vs SFT — 정성 비교

In [ ]:
sft_outputs = {p: generate(sft_model, p, **BEST_GEN) for p in eval_prompts}
comp = pd.DataFrame({
    "prompt": eval_prompts,
    "기존 KoGPT-2": [base_outputs[p][len(p):].strip() for p in eval_prompts],
    "SFT 적용":     [sft_outputs[p][len(p):].strip()  for p in eval_prompts],
})
pd.set_option("display.max_colwidth", 200)
display(comp)

> **[✍️ 작성 — 루브릭3 정성]** 지시 이행·관련성·반복/환각 관점에서 두 모델을 비교하세요.

**지시 이행** 면에서 기존 KoGPT-2는 프롬프트를 무시하고 소설·뉴스체로 이어 쓰는 경우가 많았다(예: '한우에요?' 질문에 식사 장면을 서술). 반면 SFT는 '제가 AI이기 때문에…'처럼 질문 의도에 맞춰 응답을 시작해 **관련성**이 뚜렷이 높았다. **반복·환각** 측면에서 기존 모델은 장황하게 늘어놓다 환각이 잦았고, SFT는 짧고 정돈됐지만 여전히 사실 오류가 남았다(닉슨 질문에 '존 매케인이 498세', 오헤어 공항을 '캘리포니아'로 답변). 종합하면 SFT가 지시 이행·관련성에서 확연히 우수하나, **사실 정확성은 두 모델 모두 한계**가 남는다.

## 8. (루브릭 2) Reward Model 학습

In [ ]:
from chatgpt.dataset import RewardDataset
from chatgpt.models.gpt import GPTRM
from chatgpt.trainer import RewardModelTrainer
from chatgpt.trainer.strategies import NaiveStrategy
from torch.optim import Adam

# ranking -> (chosen, rejected): ranking[0]=최고 답변 인덱스 (검증: 데이터와 83.5% 일치)
RM_PATH = "KoChatGPT/data_kochatgpt/kochatgpt_2_RM.jsonl"
rm_raw = json.load(open(RM_PATH, encoding="utf-8-sig"))
pairs = []
for d in rm_raw:
    comps = [d["completion_0"], d["completion_1"], d["completion_2"]]
    order = d["ranking"]                 # ex) [2,1,0] -> order[0]=2, completion_2가 최고
    best = comps[order[0]]
    for worse_idx in order[1:]:
        pairs.append({"prompt": d["prompt"], "chosen": best, "rejected": comps[worse_idx]})
random.shuffle(pairs)
print("RM 학습쌍 수:", len(pairs))
train_pairs = pairs[:500]; eval_pairs = pairs[500:560]

In [ ]:
strategy = NaiveStrategy()
with strategy.model_init_context():
    reward_model = GPTRM(pretrained=MODEL_NAME).to(DEVICE)

train_rm_ds = RewardDataset(train_pairs, tokenizer, max_length=256)
eval_rm_ds  = RewardDataset(eval_pairs,  tokenizer, max_length=256)

rm_optim = Adam(reward_model.parameters(), lr=5e-6)
rm_trainer = RewardModelTrainer(model=reward_model, strategy=strategy, optim=rm_optim,
                                train_dataset=train_rm_ds, eval_dataset=eval_rm_ds,
                                batch_size=4, max_epochs=1)
rm_trainer.fit(use_lora=0)     # fit()는 use_lora 인자가 필수
reward_model.eval()

## 9. (루브릭 2) SFT 모델 vs RM 모델 비교·분석

In [ ]:
def reward_score(text):
    ids = tokenizer(text, return_tensors="pt", truncation=True, max_length=256).input_ids.to(DEVICE)
    with torch.no_grad():
        s = reward_model(ids)
    return float(s.detach().cpu().numpy().reshape(-1)[0])

rows = []
for p in eval_prompts:
    rows.append({"prompt": p,
                 "SFT답변 보상": round(reward_score(sft_outputs[p]), 3),
                 "기존답변 보상": round(reward_score(base_outputs[p]), 3)})
rm_cmp = pd.DataFrame(rows)
display(rm_cmp)
print("SFT답변이 더 높은 보상 비율: %.0f%%" %
      (100*(rm_cmp["SFT답변 보상"] > rm_cmp["기존답변 보상"]).mean()))

> **[✍️ 작성 — 루브릭2 SFT vs RM]** RM은 답변을 '점수화'하는 모델입니다. RM이 SFT 답변에 더 높은 보상을 주는지(정량), 사람 판단과 일치하는지(정성)를 분석하세요.

RM은 답변에 스칼라 보상을 매기는 **'채점 모델'**이다. 학습한 RM으로 다섯 프롬프트에서 SFT 답변과 기존 KoGPT-2 답변의 보상을 비교하니, **SFT 답변이 더 높은 보상을 받은 비율이 80%(4/5)**였다. 예컨대 '불고기용 고기 한우에요?'는 SFT **2.730** vs 기존 0.736, '건강하게 살 수 있는 방법은?'은 **0.732** vs -1.630으로 격차가 컸다. 다만 '시카고 오헤어 공항' 문항은 기존 **0.383 > SFT -0.056**으로 역전됐는데, 두 답변 모두 오답(환각)이라 RM이 표면적 유창함에 점수를 준 사례로 **RM 보상이 사람 판단(사실성)과 항상 일치하지는 않음**을 보여준다. 종합하면 RM은 대체로 지시 이행·관련성이 높은 SFT 답변에 더 높은 보상을 주어 **사람 선호를 근사**한다고 볼 수 있다.

## 10. 결론 및 회고

**개선 전략 요약** — ① SFT 데이터 EDA·정제(647건 노이즈 제거)·증강, ② 디코딩 하이퍼파라미터 서치(greedy/beam/top-k/top-p 비교 후 top-k+repetition_penalty 채택), ③ BLEU·ROUGE 정량평가와 RM 보상 기반 비교를 결합했다.

**정량 결과** — 기존 KoGPT-2 대비 SFT는 BLEU 0.14→0.85, ROUGE-1/L 0.0082→0.0134로 상승했고, RM 보상 비교에서도 SFT 답변이 80%의 프롬프트에서 더 높은 보상을 받았다.

**정성 결과** — SFT는 질문 의도에 맞춘 응답 시작·간결함·관련성에서 기존 모델을 확연히 앞섰다. 다만 사실 정확성(환각)은 두 모델 모두 한계가 남았고, RM도 유창하지만 틀린 답변에 높은 점수를 주는 경우가 있었다.

**한계와 다음 시도** — (1) foundation model을 더 큰 `skt/ko-gpt-trinity-1.2B` 등으로 교체, (2) RM 보상을 실제 정책 최적화로 잇는 **PPO** 단계 추가, (3) 사실성 강화를 위한 데이터 품질·검증셋 확대, (4) 한국어 특성을 반영한 형태소 기반 평가 지표 병행을 다음 개선 방향으로 제안한다.

### 제출
노트북 전체 실행 + [✍️ 작성] 칸 작성 후 → 모두의연구소 '프로젝트 제출'에 **Colab/GitHub URL** 또는 **PDF(파일→다운로드→.pdf)** 로 제출 (ZIP/PDF/이미지 ≤5MB).